# Layer-wise density trajectory

The primary readout of this experiment. Dice establishes that the method did
not break; this is the result.

Two views are carried throughout because they can disagree:

- **density** — how connected each layer is
- **live budget share** — where the capacity actually went

Stages 0-2 hold about 6 percent of encoder parameters, so they can swing from
0.30 to 1.00 density while moving almost no budget. A density plot can look
dramatic while the budget plot is flat.

The gate conditions in `docs/preregistration.md` are frozen. This notebook
evaluates them as a factual matter and does not interpret them.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO / "src"))

from analysis.trajectory import (
    churn,
    deep_shallow_split,
    erk_reference,
    evaluate_gates,
    format_report,
    load_trajectory,
    stage_budget_shares,
    stage_densities,
    steps,
)

RESULTS = REPO / "results"
TARGET_DENSITY = 0.30

runs = {p.parent.name: p for p in sorted(RESULTS.glob("*/trajectory.csv"))}
print("runs found:", list(runs))

In [ ]:
# The arm the experiment is about. Change to inspect another.
RUN = "rigl_seed0" if "rigl_seed0" in runs else next(iter(runs))
rows = load_trajectory(runs[RUN])
all_steps = steps(rows)
print(f"{RUN}: {len(rows)} rows, {len(all_steps)} checkpoints, "
      f"steps {all_steps[0]}..{all_steps[-1]}")

erk_dens, erk_share = erk_reference(rows, TARGET_DENSITY)
print("\nERK null (the task-independent prior):")
for s in sorted(erk_dens):
    print(f"  stage{s}: density {erk_dens[s]:.3f}   budget {100*erk_share[s]:5.2f}%")

## Density trajectory, against the ERK null

Dashed lines are the ERK allocation. Drift toward them is a rediscovery of a
known prior; departure from them is the task-specific signal Gate B asks for.

In [ ]:
stages = sorted({r["stage"] for r in rows})
colors = plt.cm.viridis(np.linspace(0, 0.9, len(stages)))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for c, s in zip(colors, stages):
    ax = axes[0]
    ax.plot(all_steps, [stage_densities(rows, st)[s] for st in all_steps],
            color=c, label=f"stage {s}")
    ax.axhline(erk_dens[s], color=c, ls="--", lw=0.8, alpha=0.6)
axes[0].axhline(TARGET_DENSITY, color="k", ls=":", lw=1, label="uniform 0.30")
axes[0].set(xlabel="step", ylabel="density", title="Stage density (dashed = ERK)")
axes[0].legend(fontsize=8, ncol=2)

for c, s in zip(colors, stages):
    ax = axes[1]
    ax.plot(all_steps, [100 * stage_budget_shares(rows, st).get(s, 0) for st in all_steps],
            color=c, label=f"stage {s}")
    ax.axhline(100 * erk_share[s], color=c, ls="--", lw=0.8, alpha=0.6)
axes[1].set(xlabel="step", ylabel="share of live parameters (%)",
            title="Capacity allocation (dashed = ERK)")
axes[1].legend(fontsize=8, ncol=2)

fig.suptitle(f"{RUN}: density vs capacity. These can disagree.")
fig.tight_layout()

## Shallow / deep budget split

The headline figure. ERK moves 9.9 points of budget from the deep stages to
the shallow ones (93.9 to 84.0 percent). That is the full size of the null's
effect on capacity allocation.

In [ ]:
split = [deep_shallow_split(stage_budget_shares(rows, st)) for st in all_steps]
erk_shallow, erk_deep = deep_shallow_split(erk_share)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(all_steps, [d for _, d in split], label="deep (stages 3-5)", color="C3")
ax.plot(all_steps, [s for s, _ in split], label="shallow (stages 0-2)", color="C0")
ax.axhline(erk_deep, color="C3", ls="--", lw=0.9, label=f"ERK deep {erk_deep:.1f}%")
ax.axhline(erk_shallow, color="C0", ls="--", lw=0.9, label=f"ERK shallow {erk_shallow:.1f}%")
ax.axhline(93.9, color="k", ls=":", lw=0.8, label="uniform 93.9 / 6.1%")
ax.axhline(6.1, color="k", ls=":", lw=0.8)
ax.set(xlabel="step", ylabel="share of live parameters (%)",
       title=f"{RUN}: where the capacity sits")
ax.legend(fontsize=8)
fig.tight_layout()

## Churn

The disambiguator for a static allocation. A mistuned drop fraction gives low
churn from step one; a converged mask gives high churn early that decays.
Same tail, different history, so read the **early** segment.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
for c, s in zip(colors, stages):
    layer_names = sorted({r["layer_name"] for r in rows if r["stage"] == s})
    series = []
    for st in all_steps:
        ch = churn(rows, st)
        vals = [ch[n] for n in layer_names if n in ch]
        series.append(np.mean(vals) if vals else np.nan)
    ax.plot(all_steps, series, color=c, label=f"stage {s}")
ax.set(xlabel="step", ylabel="(pruned + regrown) / live",
       title=f"{RUN}: mask churn per logging interval")
ax.legend(fontsize=8, ncol=2)
fig.tight_layout()

## Regrowth informativeness

Top-k overlap between regrowth scores computed on a foreground-oversampled
batch and a background-dominated batch, at the same masked positions.

Low overlap means the regrowth criterion is reading the batch rather than the
task. That is a third candidate explanation for a null trajectory, distinct
from a mistuned drop fraction and from ERK being correct, and it points at the
foreground oversampling rate. **Logged, not interpreted.**

In [ ]:
import csv

probe_path = RESULTS / RUN / "regrowth_informativeness.csv"
if probe_path.exists():
    probe = list(csv.DictReader(open(probe_path)))
    if probe:
        fig, ax = plt.subplots(figsize=(7, 4.5))
        for st in sorted({int(r["step"]) for r in probe}):
            at = [r for r in probe if int(r["step"]) == st]
            at.sort(key=lambda r: (int(r["stage"]), r["layer_name"]))
            ax.plot([r["layer_name"] for r in at],
                    [float(r["topk_overlap_fg_vs_bg"]) for r in at],
                    marker="o", label=f"step {st}")
        ax.set(ylabel="top-k overlap, foreground vs background batch",
               ylim=(0, 1), title=f"{RUN}: is the regrowth signal task-driven?")
        ax.tick_params(axis="x", rotation=60)
        ax.legend(fontsize=8)
        fig.tight_layout()
    else:
        print("probe file is empty")
else:
    print(f"no probe file at {probe_path}")

## Gate conditions

Frozen in `docs/preregistration.md`. Evaluated below as a factual matter.
Interpretation follows the protocol in that document; it is not made here.

In [ ]:
print(format_report(evaluate_gates(rows, target_density=TARGET_DENSITY)))

## All arms together

`static_sparse` is the control that separates dynamic reallocation from the
network merely being overparameterized. Its trajectory should be flat by
construction; if it is not, something is wrong with the masking.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for run, path in runs.items():
    r = load_trajectory(path)
    st = steps(r)
    ax.plot(st, [deep_shallow_split(stage_budget_shares(r, s))[1] for s in st],
            label=run)
ax.axhline(erk_deep, color="k", ls="--", lw=0.9, label=f"ERK deep {erk_deep:.1f}%")
ax.set(xlabel="step", ylabel="deep-stage share of live parameters (%)",
       title="Capacity in the deep stages, all arms")
ax.legend(fontsize=8)
fig.tight_layout()